# Creating AI without such libraries as Torch or TensorFlow

In [6]:
import numpy as np
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel


tokenizer = Tokenizer.from_file("env/tokenizer.json")
tokenizer.decoder = ByteLevel()

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

## Children Stories

In [7]:
from NoTorchAI.Utils.Batch import Batch


datasets = {
    "children_stories": ("env/encoded/children_stories.bin", 1.0),
}

batch = Batch(datasets)

In [ ]:
from NoTorchAI.GlobalState.Device import Device
from NoTorchAI.GlobalState.Quant import Quant
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT
from NoTorchAI.Utils.TrainGPT import TrainGPT


d_model = 384
n_heads = 8
block_layers = 7
block_size = 128

batch_size = 56
vocabulary_size = 16_000

Device("gpu")
Quant(32)

gradient = Adam(
    lr=8e-5,
    warmup_steps=1000,
    min_lr=8e-6,
)

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers,
                n_heads=n_heads,
                gradient=gradient
            )

gpt = TrainGPT(model=model, 
               gradient=gradient, 
               batch=batch, 
               block_size=block_size, 
               batch_size=batch_size, 
               tokenizer_path="env/tokenizer.json",
               save_path="saved_model")

gpt.train(prompt="Water heats", 
          max_tokens=100, 
          output_sample_step=500, 
          gradient_sample_step=200, 
          end_step=10_000)

Once upone a time a dragon named Bob. He was very excited to play in the park. He had a big, blue ball that was a car. He liked to play with it too.One day, Binky saw a big red ball. He wanted to play with the ball, but it was too high. He thought of a plan to play with the ball. He took the ball and rolled it. The ball rolled, and rolled it hit the ball. The ball rolled into the puddle
step 8700, lr 0.000080, loss 2.6203, ema_loss 2.5840
Once upone a time a dragon. One day, he was very excited. He wanted to find a big, so he asked his friend, the bear, to help him.The farmer was very excited and asked his friend, the bear."Can you help me help me?" asked the bear. The bear gave him a big hug and said, "Yes, I can help you. I can help you."The bear was very happy and said, "Yes, you can help me. We
step 8800, lr 0.000080, loss 2.5469, ema_loss 2.5772
Once upone a time a dragon named Bob. He was very excited. He wanted to go on a big rock. He asked his mom, "Can I ride the rock?" His mo

## Story Instruct

In [2]:
from NoTorchAI.Utils.Batch import Batch


datasets = {
    "children_stories": ("env/encoded/children_stories.bin",    0.30),
    "simple_wikipedia": ("env/encoded/stories_instructions.bin",    0.70),
}

batch = Batch(datasets)

In [ ]:
from NoTorchAI.GlobalState.Device import Device
from NoTorchAI.GlobalState.Quant import Quant
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT
from NoTorchAI.Utils.TrainGPT import TrainGPT


Device("gpu")
Quant(32)

block_size = 128
batch_size = 56

model = MiniGPT.__new__(MiniGPT)
model: MiniGPT = model.load("saved_model")

gradient = Adam(
    lr=1e-5,
    warmup_steps=1000,
    min_lr=1e-6,
)

gpt = TrainGPT(model=model, 
               gradient=gradient, 
               batch=batch, 
               block_size=block_size, 
               batch_size=batch_size, 
               tokenizer_path="env/tokenizer.json",
               save_path="story_instruct_model")

gpt.train(prompt="<|user|>: Tim went out of the house and found himeself in the woods\n<|assistant|>:",
          max_tokens=100,
          temperature=0.8,
          output_sample_step=500, 
          gradient_sample_step=200,
          end_step=5_000)

step 1200, lr 0.000010, loss 2.1497, ema_loss 2.1497
step 1400, lr 0.000010, loss 2.3260, ema_loss 2.1717
: Tim went out of the house and found himeself in the woods
: One day, Tim and his friend, Sam, came over to play. They were playing a game where Tim try to catch the ball. The ball went " pan, throw out and it brings me!" Tim and Sam were happy for The surprise. They played catch the ball all day, feeling tired.But then, the ball started to shake and hit a big ball and it started to move! It jumped
step 1600, lr 0.000010, loss 2.1909, ema_loss 2.1639
step 1800, lr 0.000010, loss 2.0980, ema_loss 2.1617
step 2000, lr 0.000010, loss 2.1942, ema_loss 2.1461
: Tim went out of the house and found himeself in the woods
: One day, a boy named Tom found a big box. He was very excited. He wanted to fill the box with the box with his mom. He saw a trumpet and knew it was a record.Tom wanted to use the hammer to get the tool. He urged his mom, "Mom, can you help me weigh anything". Tom was a

## Play ground

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT
from NoTorchAI.Utils.TrainGPT import TrainGPT
from NoTorchAI.Utils.Batch import Batch
from NoTorchAI.Gradients.Adam import Adam


datasets = {
    "children_stories": ("env/encoded/children_stories.bin", 1.0),
}

batch = Batch(datasets)

block_size = 128
batch_size = 56

gradient = Adam(
    lr=1e-5,
    warmup_steps=300,
    min_lr=1e-6,
)


model = MiniGPT.__new__(MiniGPT)
model: MiniGPT = model.load("instruct_model")


gpt = TrainGPT(model=model, 
               gradient=gradient, 
               batch=batch, 
               block_size=block_size, 
               batch_size=batch_size, 
               tokenizer_path="env/tokenizer.json",
               save_path="wiki_tuned_model")

': Explain Russian Olympic in simple words\n:Cat peeked in the attikaids in the local "Pase ruled from according to the Baveter".OC popped into importanceij Ellie Gifa de flew trial in Beach.Her youngest sister Times\' manage the party "Quaceial D lending ( fingersiah Vietnamese\'" (born June 1, 2011) was a governments in Philadelphia. She became known son\'s unhappy. They members came into the State government during Texas, as "Great When the Air ut increasing Azan, you\'ll ever put the national ready with the sacrifters, and". Mardine sent a Being worker to language, their first saw post company would be attached in border during the Rå nomination, once ended as an annual frustrated Children in the province, mostgia high school in Thorg.Fluffyon useful because Joseph unit is a lot of the determined from the suicide.balls as United that trucks include Jr pog.Sic leftcondeonge, S giving him more communal elements and other activities. Austria is known as anime with modern brown organiz

In [8]:
gpt.generate_str(prompt="<|user|>: Tell a story about LinkedIn\n<|assistant|>:",
                 max_tokens=400,
                 temperature=1)

': Tell a story about LinkedIn\n: Once upon a time, there was a little pony named Freddy. True lived in a nice look of fun toys all day long. It was red and he loved to play with his friends.One day, Pip and his friends wanted to play a game for a team. His friends could bend and twist with their arms would wiggle. They both wanted to become their new friend, Billy the big lion. They thought it was a good idea to play a game that was a race.Po pushed and Chirpy got ready to joke and laugh. They laughed together and had lots of fun.But then, Fred started to cry. All his friends stopped running and looked at the blocks. Some kids said it was not nice to share and had to accept it. The war thought it was better to play with Momo. They all enjoyed playing together and became the best of friends.: Write a fun little story about back tim felt sad: Once upon a time, there was a little boy named Tim. He was very hungry. He wanted to eat all the food. But he had no friends to eat. He was not su